In [1]:
"""
03. Data Preprocessing
Emotion Classification

Purpose:
    Create leakage-free Train / Validation / Test splits
    while preserving class distribution as much as possible.
"""


from pathlib import Path

from utils.config import (
    SEED,
    TEXT_COL,
    LABEL_COL
)

from utils.data import (
    load_data,
    remove_exact_duplicates,
    create_text_groups,
    split_data,
    get_split_data
)


# =========================================================
# Configuration
# =========================================================

SPLIT_DIR = Path(
    "results/splits"
)

SPLIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =========================================================
# Header
# =========================================================

print("=" * 70)
print("03. Data Preprocessing")
print("=" * 70)

print()


# =========================================================
# 1. Load Dataset
# =========================================================

print("[1] Load Dataset")
print("-" * 70)

df = load_data()

print(
    f"Dataset size : {len(df):,}"
)

print()


# =========================================================
# 2. Remove Exact Duplicate Rows
# =========================================================

print("[2] Remove Exact Duplicate Rows")
print("-" * 70)

df = remove_exact_duplicates(df)


# =========================================================
# 3. Create Text Groups
# =========================================================

print("[3] Create Text Groups")
print("-" * 70)

df = create_text_groups(df)

num_groups = (
    df["text_group"]
    .nunique()
)

print(
    f"Rows          : {len(df):,}"
)

print(
    f"Unique texts   : {num_groups:,}"
)

print(
    f"Duplicate text groups : "
    f"{len(df) - num_groups:,}"
)

print()


# =========================================================
# 4. Split Dataset
# =========================================================

print("[4] Create Train / Validation / Test Split")
print("-" * 70)

df = split_data(
    df,
    seed=SEED
)

print(
    df["split"]
    .value_counts()
)

print()


# =========================================================
# 5. Get Split Data
# =========================================================

(
    train_df,
    val_df,
    test_df
) = get_split_data(df)


# =========================================================
# 6. Split Size
# =========================================================

print("[5] Split Size")
print("-" * 70)

total = len(df)

split_sizes = {
    "Train": len(train_df),
    "Validation": len(val_df),
    "Test": len(test_df)
}

for name, size in split_sizes.items():

    ratio = (
        size / total * 100
    )

    print(
        f"{name:<12}: "
        f"{size:>8,} "
        f"({ratio:>6.2f}%)"
    )

print()


# =========================================================
# 7. Label Distribution
# =========================================================

print("[6] Label Distribution")
print("-" * 70)

print(
    "Train:"
)

print(
    train_df[LABEL_COL]
    .value_counts()
    .sort_index()
)

print()

print(
    "Validation:"
)

print(
    val_df[LABEL_COL]
    .value_counts()
    .sort_index()
)

print()

print(
    "Test:"
)

print(
    test_df[LABEL_COL]
    .value_counts()
    .sort_index()
)

print()


# =========================================================
# 8. Label Ratio
# =========================================================

print("[7] Label Ratio (%)")
print("-" * 70)

for name, split_df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:

    ratio = (
        split_df[LABEL_COL]
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    )

    print(
        f"{name}:"
    )

    print(ratio)

    print()


# =========================================================
# 9. Leakage Check
# =========================================================

print("[8] Data Leakage Check")
print("-" * 70)


train_texts = set(
    train_df[TEXT_COL]
)

val_texts = set(
    val_df[TEXT_COL]
)

test_texts = set(
    test_df[TEXT_COL]
)


train_val_overlap = (
    train_texts & val_texts
)

train_test_overlap = (
    train_texts & test_texts
)

val_test_overlap = (
    val_texts & test_texts
)


print(
    "Train ↔ Validation:",
    len(train_val_overlap)
)

print(
    "Train ↔ Test      :",
    len(train_test_overlap)
)

print(
    "Validation ↔ Test :",
    len(val_test_overlap)
)

print()


# =========================================================
# Leakage Assertion
# =========================================================

assert len(train_val_overlap) == 0
assert len(train_test_overlap) == 0
assert len(val_test_overlap) == 0


print(
    "✓ No duplicate text exists across splits."
)

print()


# =========================================================
# 10. Group Leakage Check
# =========================================================

print("[9] Group Leakage Check")
print("-" * 70)


train_groups = set(
    train_df["text_group"]
)

val_groups = set(
    val_df["text_group"]
)

test_groups = set(
    test_df["text_group"]
)


assert train_groups.isdisjoint(
    val_groups
)

assert train_groups.isdisjoint(
    test_groups
)

assert val_groups.isdisjoint(
    test_groups
)


print(
    "✓ All text groups are isolated "
    "within a single split."
)

print()


# =========================================================
# 11. Save Split Data
# =========================================================

print("[10] Save Split Data")
print("-" * 70)


train_path = (
    SPLIT_DIR / "train.csv"
)

val_path = (
    SPLIT_DIR / "validation.csv"
)

test_path = (
    SPLIT_DIR / "test.csv"
)


train_df.to_csv(
    train_path,
    index=False
)

val_df.to_csv(
    val_path,
    index=False
)

test_df.to_csv(
    test_path,
    index=False
)


print(
    f"Train      : {train_path}"
)

print(
    f"Validation : {val_path}"
)

print(
    f"Test       : {test_path}"
)

print()


# =========================================================
# Complete
# =========================================================

print("=" * 70)
print("Data Preprocessing Complete")
print("=" * 70)

03. Data Preprocessing

[1] Load Dataset
----------------------------------------------------------------------
Dataset size : 436,809

[2] Remove Exact Duplicate Rows
----------------------------------------------------------------------
Exact Duplicate Removal
----------------------------------------------------------------------
Before : 436,809
After  : 436,122
Removed: 687

[3] Create Text Groups
----------------------------------------------------------------------
Rows          : 436,122
Unique texts   : 393,822
Duplicate text groups : 42,300

[4] Create Train / Validation / Test Split
----------------------------------------------------------------------
split
train         305345
validation     65422
test           65355
Name: count, dtype: int64

[5] Split Size
----------------------------------------------------------------------
Train       :  305,345 ( 70.01%)
Validation  :   65,422 ( 15.00%)
Test        :   65,355 ( 14.99%)

[6] Label Distribution
------------------------